In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import cohen_kappa_score

from config.config import REGISTRIES_DIR, SUBREGISTRIES_DIR, VersionObject

DOCUMENT_REGISTRY: Path =  REGISTRIES_DIR / "document_registry.csv"
BASE_REGISTRY: Path = REGISTRIES_DIR / "base_output_registry.csv"
RAW_REGISTRY: Path = SUBREGISTRIES_DIR / "raw_file_output_registry.csv"
EXTRACTION_REGISTRY: Path = SUBREGISTRIES_DIR / "extraction_output_registry.csv"
CLASSIFICATION_REGISTRY: Path = SUBREGISTRIES_DIR / "DAS_classification_output_registry.csv"

version_object = VersionObject(pipeline_version="v1.0.0", software_version="v1.0.2")
df_document_registry = pd.read_csv(DOCUMENT_REGISTRY)
df_base_registry = pd.read_csv(BASE_REGISTRY)
df_raw_registry = pd.read_csv(RAW_REGISTRY)
df_extraction_registry = pd.read_csv(EXTRACTION_REGISTRY)
df_classification_registry = pd.read_csv(CLASSIFICATION_REGISTRY)

merge_base_classification = df_classification_registry.merge(df_base_registry[["output_sha", "doc_doi", "software_version"]], on="output_sha", how="left")

df_haiku_v2 = merge_base_classification.loc[(merge_base_classification["model"]=="claude-haiku-4-5-20251001")&
                                            (merge_base_classification["software_version"]=="v1.0.2"), 
                                            ["output_sha", "doc_doi", "text"]]



In [ ]:
df_haiku_v2.shape

In [ ]:
df_haiku_v2_validation_sample = df_haiku_v2.sample(n=500, random_state=42)

df_haiku_v2_validation_sample.to_csv("data/metadata/annotations/validation_sample_haiku_v2.csv")

df_haiku_v2_validation_sample.duplicated().sum()

In [ ]:
df_haiku_v2_labels = merge_base_classification.loc[(merge_base_classification["model"]=="claude-haiku-4-5-20251001")&
                                            (merge_base_classification["software_version"]=="v1.0.2"), 
                                            ["output_sha", "doc_doi", "label", "text"]]

df_haiku_incorrect = df_haiku_v2_labels.loc[df_haiku_v2_labels["label"]=="incorrect"]

df_haiku_partial_access = df_haiku_v2_labels.loc[df_haiku_v2_labels["label"]=="partial_access"]

df_haiku_no_access = df_haiku_v2_labels.loc[df_haiku_v2_labels["label"]=="no_access"]

display(df_haiku_incorrect)
display(df_haiku_no_access) 
display(df_haiku_partial_access)

In [ ]:
df_haiku_v2_manual_annotations = pd.read_csv("data/metadata/annotations/DAS_annotated_sample_500.csv")



In [ ]:
df_haiku_v2_manual_annotations = df_haiku_v2_manual_annotations.drop(columns=["Unnamed: 0"])
print(df_haiku_v2_manual_annotations.columns.tolist())


In [ ]:

"""TESTING MANUAL-CLASSIFIER AGREEMENT WITH COHEN'S KAPPA"""
# get classifier labels for the same shas
classifier_labels = df_classification_registry[
    df_classification_registry["output_sha"].isin(df_haiku_v2_manual_annotations["output_sha"])
][["output_sha", "label"]].rename(columns={"label": "label_classifier"})

# merge with your manual labels
df_validation = df_haiku_v2_manual_annotations[["output_sha", "label"]].rename(columns={"label": "label_manual"})
df_validation = df_validation.merge(classifier_labels, on="output_sha", how="inner")

# sanity check — should be 500
print(f"Matched rows: {len(df_validation)}")
print(df_validation[["label_manual", "label_classifier"]].value_counts())

# compute kappa
kappa = cohen_kappa_score(df_validation["label_manual"], df_validation["label_classifier"])
print(f"\nCohen's Kappa: {kappa:.4f}")

In [ ]:
"""CONFUSION MATRIX FOR MANUAL-CLASSIFIER AGREEMENTS"""

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

labels = sorted(df_validation["label_manual"].unique())
cm = confusion_matrix(df_validation["label_manual"], 
                      df_validation["label_classifier"], 
                      labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(xticks_rotation=45)
plt.tight_layout()
plt.savefig("confusion_matrix_validation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
"""SCRIPT FOR EYEBALLING DISAGREEMENTS"""

df_disagreements = df_validation[
    df_validation["label_manual"] != df_validation["label_classifier"]
].merge(df_haiku_v2_manual_annotations[["output_sha", "doc_doi", "text"]], on="output_sha", how="left")

print(f"Total disagreements: {len(df_disagreements)}\n")
for _, row in df_disagreements.iterrows():
    print(f"DOI:        {row['doc_doi']}")
    print(f"SHA:        {row['output_sha']}")
    print(f"Manual:     {row['label_manual']}")
    print(f"Classifier: {row['label_classifier']}")
    print(f"Text:       {row['text']}")
    print("-" * 80)

In [ ]:
import numpy as np
import scipy.stats as stats


def kappa_ci(kappa, n, alpha=0.05):
    se = np.sqrt((kappa * (1 - kappa)) / n)
    z = stats.norm.ppf(1 - alpha / 2)
    return kappa - z * se, kappa + z * se

lower, upper = kappa_ci(0.9759, 500)
print(f"95% CI: [{lower:.4f}, {upper:.4f}]")

In [ ]:
df_haiku_v2_manual_annotations.to_csv("data/metadata/annotations/project-1-at-2026-03-24-14-41-b206434f.csv", index=False)